# Visualising elfe3D_GPR Results and Post Processing

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import os
import pyvista as pv
from scipy.interpolate import griddata

In [2]:
base_folder = "F:\\Projects\\EMGeoInversion\\Tests_Thesis\\4. June"
postprocess_folder = os.path.join(base_folder, "postprocess2")
analytical_folder = os.path.join(base_folder, "semi-analytic_100MHz")

z_value = 0.01
z_tol = 0.05
xlim=(-2.25, 5.75)
ylim=(-2.25, 5.75)
xlim_base = (-5, 5)
ylim_base = (-5, 5)
xlim_a = (-1.5, 5)
ylim_a = (-1.5, 5)
z_tol=0.05
dpi=300
grid_resolution=3000
amplitude_cmap='Blues_r'
phase_cmap='RdBu_r'

component = 'Ex' 

In [3]:
# Load the data
vtk_file_1 = os.path.join(base_folder, "GPR_model_air_min_domain_ele.1.vtk")
elfe_vtk_1 = pv.read(vtk_file_1)

vtk_file_2 = os.path.join(base_folder, "GPR_model_air_l20.1.vtk")
elfe_vtk_2 = pv.read(vtk_file_2)

elfe_vtk_all = [elfe_vtk_1, elfe_vtk_2]

In [5]:
# Extracting Plot Data
amp_field_elfe = f"Real_{component}"
phase_field_elfe = f"Imag_{component}"

# --- elfe3D Cell Centers and Scalars for all entries in elfe_vtk_all ---
amp_elfe_grid_all = []
phase_elfe_grid_all = []

for elfe_vtk_entry in elfe_vtk_all:
    # --- elfe3D Cell Centers and Scalars ---
    cell_centers = elfe_vtk_entry.cell_centers().points
    real_elfe = elfe_vtk_entry.cell_data[f"Real_{component}"]
    imag_elfe = elfe_vtk_entry.cell_data[f"Imag_{component}"]
    complex_elfe = real_elfe + 1j * imag_elfe

    # --- Filter to z-slice ---
    mask = np.abs(cell_centers[:, 2] - z_value) <= z_tol
    x_elfe = cell_centers[mask, 0]
    y_elfe = cell_centers[mask, 1]
    complex_elfe_slice = complex_elfe[mask]

    xi = np.linspace(xlim[0], xlim[1], grid_resolution)
    yi = np.linspace(ylim[0], ylim[1], grid_resolution)
    grid_x, grid_y = np.meshgrid(xi, yi)

    # --- Interpolate elfe3D fields to regular grid ---
    complex_elfe_grid = griddata((x_elfe, y_elfe), complex_elfe_slice, (grid_x, grid_y), method='linear')
    amp_elfe_grid = np.log10(np.clip(np.abs(complex_elfe_grid), 1e-60, None))
    phase_elfe_grid = np.angle(complex_elfe_grid)  # returns phase in radians (−π to π)

    amp_elfe_grid_all.append(amp_elfe_grid)
    phase_elfe_grid_all.append(phase_elfe_grid)

phase_clim = [-np.pi, np.pi]
amp_min = np.floor(np.nanmin(amp_elfe_grid_all[0]))
amp_max = np.ceil(np.nanmax(amp_elfe_grid_all[0]))
amp_clim = [amp_min, amp_max]

# Analytical
# Mask grid_x and grid_y to be within xlim_a and ylim_a
mask_x = (grid_x >= xlim_a[0]) & (grid_x <= xlim_a[1])
mask_y = (grid_y >= ylim_a[0]) & (grid_y <= ylim_a[1])
mask = mask_x & mask_y

# Mask grid_x and grid_y to be within xlim_a and ylim_a, and flatten to get valid points
valid_mask = mask.flatten()
grid_x_bounded = grid_x.flatten()[valid_mask]
grid_y_bounded = grid_y.flatten()[valid_mask]

analytical_file = os.path.join(analytical_folder, "air_z0.01_100MHz_xmax5.csv")
analytical_slices = np.loadtxt(analytical_file, delimiter=',', skiprows=1)

x = analytical_slices[:, 0]
y = analytical_slices[:, 1]

x_unique = np.unique(x)
y_unique = np.unique(y)
nx, ny = len(x_unique), len(y_unique)

if len(x) != nx * ny:
    raise ValueError("Mismatch in analytical grid. Expected meshgrid-style structure.")

if component == 'Ex':
    real_index = 4
    imag_index = 5
elif component == 'Ey':
    real_index = 8
    imag_index = 9
elif component == 'Ez':
    real_index = 12
    imag_index = 13
else:
    raise ValueError("Invalid component. Choose from 'Ex', 'Ey', or 'Ez'.")

real_analytical = analytical_slices[:, real_index].reshape((ny, nx))
imag_analytical = analytical_slices[:, imag_index].reshape((ny, nx))

# ----------------------------------------------------------------------
# Extra Process Based on Inference
# ----------------------------------------------------------------------
real_analytical = -1 * real_analytical
imag_analytical = -1 * imag_analytical

complex_analytical = real_analytical + 1j * imag_analytical

# Interpolation
complex_analytical_grid = griddata((x, y), complex_analytical.flatten(), (grid_x_bounded, grid_y_bounded), method='linear')
amp_analytical_grid = np.log10(np.clip(np.abs(complex_analytical_grid), 1e-60, None))
phase_analytical_grid = np.angle(complex_analytical_grid)  # returns phase in radians (−π to π)
real_analytical_grid = griddata((x, y), real_analytical.flatten(), (grid_x_bounded, grid_y_bounded), method='linear')
imag_analytical_grid = griddata((x, y), imag_analytical.flatten(), (grid_x_bounded, grid_y_bounded), method='linear')

# Resize analytical grids to match grid_x and grid_y shape, filling extra entries with NaN
amp_analytical_grid_full = np.full(grid_x.shape, np.nan)
phase_analytical_grid_full = np.full(grid_x.shape, np.nan)
real_analytical_grid_full = np.full(grid_x.shape, np.nan)
imag_analytical_grid_full = np.full(grid_x.shape, np.nan)

amp_analytical_grid_full.flat[valid_mask] = amp_analytical_grid
phase_analytical_grid_full.flat[valid_mask] = phase_analytical_grid
real_analytical_grid_full.flat[valid_mask] = real_analytical_grid
imag_analytical_grid_full.flat[valid_mask] = imag_analytical_grid

amp_analytical_grid = amp_analytical_grid_full
phase_analytical_grid = phase_analytical_grid_full
real_analytical_grid = real_analytical_grid_full
imag_analytical_grid = imag_analytical_grid_full

# --- Plotting ---
n_entries = len(elfe_vtk_all)  # Number of entries in elfe_vtk_all
fig, axs = plt.subplots(2, n_entries + 1, figsize=(6.5 * (n_entries + 1), 12), dpi=dpi)

amp_text = r'$\log_{10}(|' + component + r'|)$: '
amp_text_list = [r'$\lambda/4$ PML and z, $\lambda/20$ Elements',
                 r'$\lambda/4$ PML, Source Box $\lambda/20$ Elements'
                 ]
phase_text = 'Phase: '
phase_text_list = amp_text_list.copy()

for i in range(n_entries):
    amp_text_list[i] = amp_text + amp_text_list[i]
    phase_text_list[i] = phase_text + phase_text_list[i]

for i, (amp_elfe_grid, phase_elfe_grid) in enumerate(zip(amp_elfe_grid_all, phase_elfe_grid_all)):
    # Amplitude elfe3D
    im0 = axs[0, i].imshow(amp_elfe_grid, extent=(xlim[0], xlim[1], ylim[0], ylim[1]), origin='lower',
            cmap=amplitude_cmap, vmin=amp_min, vmax=amp_max)
    axs[0, i].set_title(amp_text_list[i], fontsize=20, fontweight='bold')
    axs[0, i].set_xlabel("x", fontsize=18)
    axs[0, i].set_ylabel("y", fontsize=18)
    axs[0, i].tick_params(axis='both', which='major', labelsize=18)
    if i == 0:
        rect1 = plt.Rectangle((-1.5, -1.5), 6.5, 3, linewidth=4, edgecolor='red', facecolor='none', linestyle='--')
    else:
        rect1 = plt.Rectangle((-1.5, -1.5), 6.5, 6.5, linewidth=4, edgecolor='red', facecolor='none', linestyle='--')
    axs[0, i].add_patch(rect1)
    cbar0 = fig.colorbar(im0, ax=axs[0, i], fraction=0.046, pad=0.02)
    cbar0.ax.tick_params(labelsize=18)

    # Phase elfe3D
    im1 = axs[1, i].imshow(phase_elfe_grid, extent=(xlim[0], xlim[1], ylim[0], ylim[1]), origin='lower',
            cmap=phase_cmap, vmin=phase_clim[0], vmax=phase_clim[1])
    axs[1, i].set_title(phase_text_list[i], fontsize=20, fontweight='bold')
    axs[1, i].set_xlabel("x", fontsize=18)
    axs[1, i].set_ylabel("y", fontsize=18)
    axs[1, i].tick_params(axis='both', which='major', labelsize=18)
    if i == 0:
        rect2 = plt.Rectangle((-1.5, -1.5), 6.5, 3, linewidth=4, edgecolor='black', facecolor='none', linestyle='--')
    else:
        rect2 = plt.Rectangle((-1.5, -1.5), 6.5, 6.5, linewidth=4, edgecolor='black', facecolor='none', linestyle='--')
    axs[1, i].add_patch(rect2)
    cbar1 = fig.colorbar(im1, ax=axs[1, i], fraction=0.046, pad=0.02)
    cbar1.ax.tick_params(labelsize=18)

# Analytical solution (rightmost column)
im2 = axs[0, n_entries].imshow(amp_analytical_grid, extent=(xlim[0], xlim[1], ylim[0], ylim[1]), origin='lower',
    cmap=amplitude_cmap, vmin=amp_min, vmax=amp_max)
axs[0, n_entries].set_title('Analytical Solution', fontsize=20, fontweight='bold')
axs[0, n_entries].set_xlabel("x", fontsize=18)
axs[0, n_entries].set_ylabel("y", fontsize=18)
axs[0, n_entries].tick_params(axis='both', which='major', labelsize=18)
rect1 = plt.Rectangle((-1.5, -1.5), 6.5, 6.5, linewidth=4, edgecolor='red', facecolor='none', linestyle='--')
axs[0, n_entries].add_patch(rect1)
cbar2 = fig.colorbar(im2, ax=axs[0, n_entries], fraction=0.046, pad=0.02)
cbar2.ax.tick_params(labelsize=18)

im3 = axs[1, n_entries].imshow(phase_analytical_grid, extent=(xlim[0], xlim[1], ylim[0], ylim[1]), origin='lower',
    cmap=phase_cmap, vmin=phase_clim[0], vmax=phase_clim[1])
axs[1, n_entries].set_title('Analytical Solution', fontsize=20, fontweight='bold')
axs[1, n_entries].set_xlabel("x", fontsize=18)
axs[1, n_entries].set_ylabel("y", fontsize=18)
axs[1, n_entries].tick_params(axis='both', which='major', labelsize=18)
rect2 = plt.Rectangle((-1.5, -1.5), 6.5, 6.5, linewidth=4, edgecolor='black', facecolor='none', linestyle='--')
axs[1, n_entries].add_patch(rect2)
cbar3 = fig.colorbar(im3, ax=axs[1, n_entries], fraction=0.046, pad=0.02)
cbar3.ax.tick_params(labelsize=18)

title = f'Fine Mesh Based Truncation, {component} Component of an Electric Dipole Field in Air, 2D Slice Taken z = {z_value:.3f} m Below the Source'
if ',' in title:
    idx = title.index(',')
    wrapped_title = title[:idx+1] + '\n' + title[idx+1:].lstrip()
else:
    wrapped_title = title
plt.tight_layout(rect=[0, 0, 1, 0.89])  # leave even more space for the title
plt.suptitle(wrapped_title, fontsize=27, fontweight='bold')

# Save figure
filename_combined = f"1_fine_mesh_comparison.png"
full_path_combined = os.path.join(postprocess_folder, filename_combined)

if not os.path.exists(postprocess_folder):
    os.makedirs(postprocess_folder)

fig.savefig(full_path_combined, dpi=dpi)
plt.close(fig)

In [ ]:
electric_file = os.path.join(base_folder, "out_air_l20", "electric_fields_receiver_line.txt")
e_data_elfe_1 = np.loadtxt(electric_file)

electric_file = os.path.join(base_folder, "out_air_min_domain_ele", "electric_fields_receiver_line.txt")
e_data_elfe_2 = np.loadtxt(electric_file)


e_data_elfe_all = [e_data_elfe_1, e_data_elfe_2]

labels = [r'$\lambda/4$ PML and z, $\lambda/20$ Elements',
                 r'$\lambda/4$ PML, Source Box $\lambda/20$ Elements'
                 ]


num_rec_bs=15
num_rec_ef=15
num_rec_ob=15


def plot_component_receiver_line(component, rec_x, rec_y, abs_data_list, phase_data_list, real_data_list, imag_data_list, type_rec, index, purpose, labels):
    _, axes = plt.subplots(2, 2, figsize=(14, 10))
    type_string = 'y' if type_rec == 'Broadside' else 'x' if type_rec == 'Endfire' else 'Radial distance along 45° from origin'

    # Style options
    colors = ['#1f77b4', '#2ca02c', '#d62728', '#6a0dad', "#0e0ace", "#ff00b3"]
    linestyles = ['--', ':', '--', ':', '--', ':']
    markers = ['o', 's', 'v', 'X', 'D', 'P']
    marker_size = 8     # Bigger markers (default is ~6)
    line_width = 1.5    # Thinner lines (default is ~2)

    for i, (abs_data, phase_data, rec_xi, rec_yi, label) in enumerate(zip(abs_data_list, phase_data_list, rec_x, rec_y, labels)):
        rec_distance = rec_yi if type_rec == 'Broadside' else rec_xi if type_rec == 'Endfire' else np.sqrt(rec_xi**2 + rec_yi**2)
        style_idx = i % len(colors)
        # Absolute value plot
        axes[0, 0].plot(rec_distance, abs_data,
                     linestyle=linestyles[style_idx],
                     marker=markers[style_idx],
                     markersize=marker_size,
                     color=colors[style_idx],
                     label=label,
                     linewidth=line_width)
        
        # Phase plot
        axes[0, 1].plot(rec_distance, phase_data,
                     linestyle=linestyles[style_idx],
                     marker=markers[style_idx],
                     markersize=marker_size,
                     color=colors[style_idx],
                     label=label,
                     linewidth=line_width)
        
        # Real part plot
        axes[1, 0].plot(rec_distance, real_data_list[i],
                     linestyle=linestyles[style_idx],
                     marker=markers[style_idx],
                     markersize=marker_size,
                     color=colors[style_idx],
                     label=label,
                     linewidth=line_width)
        
        # Imaginary part plot
        axes[1, 1].plot(rec_distance, imag_data_list[i],
                     linestyle=linestyles[style_idx],
                     marker=markers[style_idx],
                     markersize=marker_size,
                     color=colors[style_idx],
                     label=label,
                     linewidth=line_width)

    axes[0, 0].set_xlabel(f'{type_string} (m)', fontsize=12)
    axes[0, 0].set_ylabel(f'Abs({component})', fontsize=12)
    axes[0, 0].set_title(f'Absolute Value of {component} Field Component', fontsize=18)
    axes[0, 0].set_yscale('log')
    axes[0, 0].grid(True)
    axes[0, 0].legend(loc='upper right')

    axes[0, 1].set_xlabel(f'{type_string} (m)', fontsize=12)
    axes[0, 1].set_ylabel(f'Phase({component}) (degrees)', fontsize=12)
    axes[0, 1].set_title(f'Phase of {component} Field Component', fontsize=18)
    axes[0, 1].grid(True)

    axes[1, 0].set_xlabel(f'{type_string} (m)', fontsize=12)
    axes[1, 0].set_ylabel(f'Re({component})', fontsize=12)
    axes[1, 0].set_title(f'Real Part of {component} Field Component', fontsize=18)
    axes[1, 0].grid(True)

    axes[1, 1].set_xlabel(f'{type_string} (m)', fontsize=12)
    axes[1, 1].set_ylabel(f'Im({component})', fontsize=12)
    axes[1, 1].set_title(f'Imaginary Part of {component} Field Component', fontsize=18)
    axes[1, 1].grid(True)

    plt.subplots_adjust(top=0.82)
    title = r'Half-Space Model Response - $\epsilon_r = 4$' \
            '\n{} Receiver Line Data of {} Component'.format(type_rec, component)
    plt.suptitle(title, fontsize=20, fontweight='bold')
    plt.tight_layout()
    plot_name = f'{index}_{purpose}_{type_rec}.png'
    out_path = os.path.join(postprocess_folder, plot_name)
    plt.savefig(out_path, dpi=300)
    plt.close()

# Prepare data for each receiver type
rec_x_bs_list, rec_y_bs_list, abs_Ex_bs_list, phase_Ex_bs_list, re_Ex_bs_list, im_Ex_bs_list = [], [], [], [], [], []
rec_x_ef_list, rec_y_ef_list, abs_Ex_ef_list, phase_Ex_ef_list, re_Ex_ef_list, im_Ex_ef_list = [], [], [], [], [], []
rec_x_ob_list, rec_y_ob_list, abs_Ex_ob_list, phase_Ex_ob_list, re_Ex_ob_list, im_Ex_ob_list = [], [], [], [], [], []

for e_data_elfe in e_data_elfe_all:

    rec_x_elfe = e_data_elfe[:, 0]
    rec_y_elfe = e_data_elfe[:, 1]

    real_Ex_elfe = e_data_elfe[:, 4]
    imag_Ex_elfe = e_data_elfe[:, 5]
    abs_Ex_elfe = np.abs(real_Ex_elfe + 1j * imag_Ex_elfe)
    phase_Ex_elfe = np.angle(real_Ex_elfe + 1j * imag_Ex_elfe) # returns phase in radians (−π to π)

    # Broadside
    i_rx_bs = num_rec_ef
    rec_x_bs = rec_x_elfe[i_rx_bs:i_rx_bs + num_rec_bs]
    rec_y_bs = rec_y_elfe[i_rx_bs:i_rx_bs + num_rec_bs]
    abs_Ex_bs = abs_Ex_elfe[i_rx_bs:i_rx_bs + num_rec_bs]
    phase_Ex_bs = phase_Ex_elfe[i_rx_bs:i_rx_bs + num_rec_bs]
    re_Ex_bs = real_Ex_elfe[i_rx_bs:i_rx_bs + num_rec_bs]
    im_Ex_bs = imag_Ex_elfe[i_rx_bs:i_rx_bs + num_rec_bs]
    rec_x_bs_list.append(rec_x_bs)
    rec_y_bs_list.append(rec_y_bs)
    abs_Ex_bs_list.append(abs_Ex_bs)
    phase_Ex_bs_list.append(phase_Ex_bs)
    re_Ex_bs_list.append(re_Ex_bs)
    im_Ex_bs_list.append(im_Ex_bs)


    # Endfire
    i_rx_ef = 0
    rec_x_ef = rec_x_elfe[i_rx_ef:i_rx_ef + num_rec_ef]
    rec_y_ef = rec_y_elfe[i_rx_ef:i_rx_ef + num_rec_ef]
    abs_Ex_ef = abs_Ex_elfe[i_rx_ef:i_rx_ef + num_rec_ef]
    phase_Ex_ef = phase_Ex_elfe[i_rx_ef:i_rx_ef + num_rec_ef]
    re_Ex_ef = real_Ex_elfe[i_rx_ef:i_rx_ef + num_rec_ef]
    im_Ex_ef = imag_Ex_elfe[i_rx_ef:i_rx_ef + num_rec_ef]
    rec_x_ef_list.append(rec_x_ef)
    rec_y_ef_list.append(rec_y_ef)
    abs_Ex_ef_list.append(abs_Ex_ef)
    phase_Ex_ef_list.append(phase_Ex_ef)
    re_Ex_ef_list.append(re_Ex_ef)
    im_Ex_ef_list.append(im_Ex_ef)

    # Oblique
    i_rx_ob = num_rec_bs + num_rec_ef
    rec_x_ob = rec_x_elfe[i_rx_ob:i_rx_ob + num_rec_ob]
    rec_y_ob = rec_y_elfe[i_rx_ob:i_rx_ob + num_rec_ob]
    abs_Ex_ob = abs_Ex_elfe[i_rx_ob:i_rx_ob + num_rec_ob]
    phase_Ex_ob = phase_Ex_elfe[i_rx_ob:i_rx_ob + num_rec_ob]
    re_Ex_ob = real_Ex_elfe[i_rx_ob:i_rx_ob + num_rec_ob]
    im_Ex_ob = imag_Ex_elfe[i_rx_ob:i_rx_ob + num_rec_ob]
    rec_x_ob_list.append(rec_x_ob)
    rec_y_ob_list.append(rec_y_ob)
    abs_Ex_ob_list.append(abs_Ex_ob)
    phase_Ex_ob_list.append(phase_Ex_ob)
    re_Ex_ob_list.append(re_Ex_ob)
    im_Ex_ob_list.append(im_Ex_ob)

# Plot all on the same axes for each receiver type
plot_component_receiver_line('Ex', rec_x_ef_list, rec_y_ef_list, abs_Ex_ef_list, phase_Ex_ef_list, re_Ex_ef_list, im_Ex_ef_list, 'Endfire', 2, 'fine_mesh', labels)

### Halfspace

In [3]:
base_folder = "F:\\Projects\\EMGeoInversion\\Tests_Thesis\\3.b May-Final"
electric_file = os.path.join(base_folder, "out_4_half_PML_surf_feng", "electric_fields_receiver_line.txt")
e_data_elfe_1 = np.loadtxt(electric_file)

electric_file = os.path.join(base_folder, "out_4_qua_PML_surf_feng", "electric_fields_receiver_line.txt")
e_data_elfe_2 = np.loadtxt(electric_file)

base_folder = "F:\\Projects\\EMGeoInversion\\Tests_Thesis\\4. June"
electric_file = os.path.join(base_folder, "out_4_min_domain_ele", "electric_fields_receiver_line.txt")
e_data_elfe_3 = np.loadtxt(electric_file)

e_data_elfe_all = [e_data_elfe_1, e_data_elfe_2, e_data_elfe_3]

labels = [
    r'$\lambda/2$ PML, On Ground, Feng',
    r'$\lambda/4$ PML, On Ground, Feng',
    r'$\lambda/2$ PML, Min Domain, || DBC',
]

num_rec_bs=15
num_rec_ef=15
num_rec_ob=15

# Analytical Data
analytical_file = os.path.join(analytical_folder, "GPR-2001-4.csv")
analytical_lines = np.loadtxt(analytical_file, delimiter=',', skiprows=1)

r = analytical_lines[:, 0]

real_broadside = -1*analytical_lines[:, 7]
imag_broadside = -1*analytical_lines[:, 8]
complex_broadside = real_broadside + 1j * imag_broadside
abs_broadside = np.abs(complex_broadside)
phase_broadside = np.angle(complex_broadside)  # returns phase in radians (−π to π)

real_45deg = -1*analytical_lines[:, 11]
imag_45deg = -1*analytical_lines[:, 12]
complex_45deg = real_45deg + 1j * imag_45deg
abs_45deg = np.abs(complex_45deg)
phase_45deg = np.angle(complex_45deg)  # returns phase in radians (−π to π)

real_endfire = -1*analytical_lines[:, 3]
imag_endfire = -1*analytical_lines[:, 4]
complex_endfire = real_endfire + 1j * imag_endfire
abs_endfire = np.abs(complex_endfire)
phase_endfire = np.angle(complex_endfire)  # returns phase in radians (−π to π)

def plot_component_receiver_line(component, rec_x, rec_y, abs_data_list, phase_data_list, real_data_list, imag_data_list, x_an, y_an, abs_an, phase_an, real_an, imag_an, type_rec, index, purpose, labels):
    _, axes = plt.subplots(2, 2, figsize=(14, 10))
    type_string = 'y' if type_rec == 'Broadside' else 'x' if type_rec == 'Endfire' else 'Radial distance along 45° from origin'

    # Style options
    colors = ['#1f77b4', '#2ca02c', '#d62728', '#6a0dad', "#0e0ace", "#ff00b3"]
    linestyles = ['--', ':', '--', ':', '--', ':']
    markers = ['o', 's', 'v', 'X', 'D', 'P']
    marker_size = 8     # Bigger markers (default is ~6)
    line_width = 1.5    # Thinner lines (default is ~2)

    for i, (abs_data, phase_data, rec_xi, rec_yi, label) in enumerate(zip(abs_data_list, phase_data_list, rec_x, rec_y, labels)):
        rec_distance = rec_yi if type_rec == 'Broadside' else rec_xi if type_rec == 'Endfire' else np.sqrt(rec_xi**2 + rec_yi**2)
        style_idx = i % len(colors)
        # Absolute value plot
        axes[0, 0].plot(rec_distance, abs_data,
                     linestyle=linestyles[style_idx],
                     marker=markers[style_idx],
                     markersize=marker_size,
                     color=colors[style_idx],
                     label=label,
                     linewidth=line_width)
        # Phase plot
        axes[0, 1].plot(rec_distance, phase_data,
                     linestyle=linestyles[style_idx],
                     marker=markers[style_idx],
                     markersize=marker_size,
                     color=colors[style_idx],
                     label=label,
                     linewidth=line_width)
        
        # Real part plot
        axes[1, 0].plot(rec_distance, real_data_list[i],
                     linestyle=linestyles[style_idx],
                     marker=markers[style_idx],
                     markersize=marker_size,
                     color=colors[style_idx],
                     label=label,
                     linewidth=line_width)
        
        # Imaginary part plot
        axes[1, 1].plot(rec_distance, imag_data_list[i],
                     linestyle=linestyles[style_idx],
                     marker=markers[style_idx],
                     markersize=marker_size,
                     color=colors[style_idx],
                     label=label,
                     linewidth=line_width)

    # Analytical solution
    rec_distance_an = np.array(y_an) if type_rec == 'Broadside' else np.array(x_an) if type_rec == 'Endfire' else np.array(x_an)
    axes[0, 0].plot(rec_distance_an, abs_an, linestyle='-', color='black', label='Analytical Solution', linewidth=2)
    axes[0, 1].plot(rec_distance_an, phase_an, linestyle='-', color='black', label='Analytical Solution', linewidth=2)
    axes[1, 0].plot(rec_distance_an, real_an, linestyle='-', color='black', label='Analytical Real Part', linewidth=2)
    axes[1, 1].plot(rec_distance_an[2:], imag_an[2:], linestyle='-', color='black', label='Analytical Imaginary Part', linewidth=2)

    axes[0, 0].set_xlabel(f'{type_string} (m)', fontsize=12)
    axes[0, 0].set_ylabel(f'Abs({component})', fontsize=12)
    axes[0, 0].set_title(f'Absolute Value of {component} Field Component', fontsize=18)
    axes[0, 0].set_yscale('log')
    axes[0, 0].grid(True)
    axes[0, 0].legend(loc='upper right')

    axes[0, 1].set_xlabel(f'{type_string} (m)', fontsize=12)
    axes[0, 1].set_ylabel(f'Phase({component}) (degrees)', fontsize=12)
    axes[0, 1].set_title(f'Phase of {component} Field Component', fontsize=18)
    axes[0, 1].grid(True)

    axes[1, 0].set_xlabel(f'{type_string} (m)', fontsize=12)
    axes[1, 0].set_ylabel(f'Re({component})', fontsize=12)
    axes[1, 0].set_title(f'Real Part of {component} Field Component', fontsize=18)
    axes[1, 0].grid(True)

    axes[1, 1].set_xlabel(f'{type_string} (m)', fontsize=12)
    axes[1, 1].set_ylabel(f'Im({component})', fontsize=12)
    axes[1, 1].set_title(f'Imaginary Part of {component} Field Component', fontsize=18)
    axes[1, 1].grid(True)

    plt.subplots_adjust(top=0.82)
    title = r'Half-Space Model Response - $\epsilon_r = 4$' \
            '\n{} Receiver Line Data of {} Component'.format(type_rec, component)
    plt.suptitle(title, fontsize=20, fontweight='bold')
    plt.tight_layout()
    plot_name = f'{index}_{purpose}_{type_rec}.png'
    out_path = os.path.join(postprocess_folder, plot_name)
    plt.savefig(out_path, dpi=300)
    plt.close()

# Prepare data for each receiver type
rec_x_bs_list, rec_y_bs_list, abs_Ex_bs_list, phase_Ex_bs_list, re_Ex_bs_list, im_Ex_bs_list = [], [], [], [], [], []
rec_x_ef_list, rec_y_ef_list, abs_Ex_ef_list, phase_Ex_ef_list, re_Ex_ef_list, im_Ex_ef_list = [], [], [], [], [], []
rec_x_ob_list, rec_y_ob_list, abs_Ex_ob_list, phase_Ex_ob_list, re_Ex_ob_list, im_Ex_ob_list = [], [], [], [], [], []

for e_data_elfe in e_data_elfe_all:

    rec_x_elfe = e_data_elfe[:, 0]
    rec_y_elfe = e_data_elfe[:, 1]

    real_Ex_elfe = e_data_elfe[:, 4]
    imag_Ex_elfe = e_data_elfe[:, 5]
    abs_Ex_elfe = np.abs(real_Ex_elfe + 1j * imag_Ex_elfe)
    phase_Ex_elfe = np.angle(real_Ex_elfe + 1j * imag_Ex_elfe) # returns phase in radians (−π to π)

    # Broadside
    i_rx_bs = num_rec_ef
    rec_x_bs = rec_x_elfe[i_rx_bs:i_rx_bs + num_rec_bs]
    rec_y_bs = rec_y_elfe[i_rx_bs:i_rx_bs + num_rec_bs]
    abs_Ex_bs = abs_Ex_elfe[i_rx_bs:i_rx_bs + num_rec_bs]
    phase_Ex_bs = phase_Ex_elfe[i_rx_bs:i_rx_bs + num_rec_bs]
    re_Ex_bs = real_Ex_elfe[i_rx_bs:i_rx_bs + num_rec_bs]
    im_Ex_bs = imag_Ex_elfe[i_rx_bs:i_rx_bs + num_rec_bs]
    rec_x_bs_list.append(rec_x_bs)
    rec_y_bs_list.append(rec_y_bs)
    abs_Ex_bs_list.append(abs_Ex_bs)
    phase_Ex_bs_list.append(phase_Ex_bs)
    re_Ex_bs_list.append(re_Ex_bs)
    im_Ex_bs_list.append(im_Ex_bs)


    # Endfire
    i_rx_ef = 0
    rec_x_ef = rec_x_elfe[i_rx_ef:i_rx_ef + num_rec_ef]
    rec_y_ef = rec_y_elfe[i_rx_ef:i_rx_ef + num_rec_ef]
    abs_Ex_ef = abs_Ex_elfe[i_rx_ef:i_rx_ef + num_rec_ef]
    phase_Ex_ef = phase_Ex_elfe[i_rx_ef:i_rx_ef + num_rec_ef]
    re_Ex_ef = real_Ex_elfe[i_rx_ef:i_rx_ef + num_rec_ef]
    im_Ex_ef = imag_Ex_elfe[i_rx_ef:i_rx_ef + num_rec_ef]
    rec_x_ef_list.append(rec_x_ef)
    rec_y_ef_list.append(rec_y_ef)
    abs_Ex_ef_list.append(abs_Ex_ef)
    phase_Ex_ef_list.append(phase_Ex_ef)
    re_Ex_ef_list.append(re_Ex_ef)
    im_Ex_ef_list.append(im_Ex_ef)

    # Oblique
    i_rx_ob = num_rec_bs + num_rec_ef
    rec_x_ob = rec_x_elfe[i_rx_ob:i_rx_ob + num_rec_ob]
    rec_y_ob = rec_y_elfe[i_rx_ob:i_rx_ob + num_rec_ob]
    abs_Ex_ob = abs_Ex_elfe[i_rx_ob:i_rx_ob + num_rec_ob]
    phase_Ex_ob = phase_Ex_elfe[i_rx_ob:i_rx_ob + num_rec_ob]
    re_Ex_ob = real_Ex_elfe[i_rx_ob:i_rx_ob + num_rec_ob]
    im_Ex_ob = imag_Ex_elfe[i_rx_ob:i_rx_ob + num_rec_ob]
    rec_x_ob_list.append(rec_x_ob)
    rec_y_ob_list.append(rec_y_ob)
    abs_Ex_ob_list.append(abs_Ex_ob)
    phase_Ex_ob_list.append(phase_Ex_ob)
    re_Ex_ob_list.append(re_Ex_ob)
    im_Ex_ob_list.append(im_Ex_ob)

# Analytical data
# -------------------------------------
# Got the broadside and endire mixed up, so now we have to fix it by re-assigning to the right
# variables.
# -------------------------------------
x_an_ef = r[1:]
y_an_ef = np.zeros_like(r[1:])
abs_Ex_an_ef = abs_endfire[1:]
phase_Ex_an_ef = phase_endfire[1:]
real_Ex_an_ef = real_endfire[1:]
imag_Ex_an_ef = imag_endfire[1:]

x_an_bs = np.zeros_like(r[1:])
y_an_bs = r[1:]
abs_Ex_an_bs = abs_broadside[1:]
phase_Ex_an_bs = phase_broadside[1:]
real_Ex_an_bs = real_broadside[1:]
imag_Ex_an_bs = imag_broadside[1:]

x_an_ob = r[1:]
y_an_ob = r[1:]
abs_Ex_an_ob = abs_45deg[1:]
phase_Ex_an_ob = phase_45deg[1:]
real_Ex_an_ob = real_45deg[1:]
imag_Ex_an_ob = imag_45deg[1:]

# Plot all on the same axes for each receiver type
plot_component_receiver_line('Ex', rec_x_ef_list, rec_y_ef_list, abs_Ex_ef_list, phase_Ex_ef_list, re_Ex_ef_list, im_Ex_ef_list,
                             x_an_ef, y_an_ef, abs_Ex_an_ef, phase_Ex_an_ef, real_Ex_an_ef, imag_Ex_an_ef, 'Endfire', 3, '4_emp_mesh', labels)


In [5]:
base_folder = "F:\\Projects\\EMGeoInversion\\Tests_Thesis\\3.b May-Final"
electric_file = os.path.join(base_folder, "out_4_9_half_PML_surf_feng", "electric_fields_receiver_line.txt")
e_data_elfe_1 = np.loadtxt(electric_file)

electric_file = os.path.join(base_folder, "out_4_9_qua_PML_surf_feng", "electric_fields_receiver_line.txt")
e_data_elfe_2 = np.loadtxt(electric_file)

base_folder = "F:\\Projects\\EMGeoInversion\\Tests_Thesis\\4. June"
electric_file = os.path.join(base_folder, "out_4_9_min_domain_ele", "electric_fields_receiver_line.txt")
e_data_elfe_3 = np.loadtxt(electric_file)

e_data_elfe_all = [e_data_elfe_1, e_data_elfe_2, e_data_elfe_3]

labels = [
    r'$\lambda/2$ PML, On Ground, Feng',
    r'$\lambda/4$ PML, On Ground, Feng',
    r'$\lambda/2$ PML, Min Domain, || DBC',
]

num_rec_bs=15
num_rec_ef=15
num_rec_ob=15

# Analytical Data
analytical_file = os.path.join(analytical_folder, "GPR-2001-49.csv")
analytical_lines = np.loadtxt(analytical_file, delimiter=',', skiprows=1)

r = analytical_lines[:, 0]

real_broadside = -1*analytical_lines[:, 7]
imag_broadside = -1*analytical_lines[:, 8]
complex_broadside = real_broadside + 1j * imag_broadside
abs_broadside = np.abs(complex_broadside)
phase_broadside = np.angle(complex_broadside)  # returns phase in radians (−π to π)

real_45deg = -1*analytical_lines[:, 11]
imag_45deg = -1*analytical_lines[:, 12]
complex_45deg = real_45deg + 1j * imag_45deg
abs_45deg = np.abs(complex_45deg)
phase_45deg = np.angle(complex_45deg)  # returns phase in radians (−π to π)

real_endfire = -1*analytical_lines[:, 3]
imag_endfire = -1*analytical_lines[:, 4]
complex_endfire = real_endfire + 1j * imag_endfire
abs_endfire = np.abs(complex_endfire)
phase_endfire = np.angle(complex_endfire)  # returns phase in radians (−π to π)

def plot_component_receiver_line(component, rec_x, rec_y, abs_data_list, phase_data_list, real_data_list, imag_data_list, x_an, y_an, abs_an, phase_an, real_an, imag_an, type_rec, index, purpose, labels):
    _, axes = plt.subplots(2, 2, figsize=(14, 10))
    type_string = 'y' if type_rec == 'Broadside' else 'x' if type_rec == 'Endfire' else 'Radial distance along 45° from origin'

    # Style options
    colors = ['#1f77b4', '#2ca02c', '#d62728', '#6a0dad', "#0e0ace", "#ff00b3"]
    linestyles = ['--', ':', '--', ':', '--', ':']
    markers = ['o', 's', 'v', 'X', 'D', 'P']
    marker_size = 8     # Bigger markers (default is ~6)
    line_width = 1.5    # Thinner lines (default is ~2)

    for i, (abs_data, phase_data, rec_xi, rec_yi, label) in enumerate(zip(abs_data_list, phase_data_list, rec_x, rec_y, labels)):
        rec_distance = rec_yi if type_rec == 'Broadside' else rec_xi if type_rec == 'Endfire' else np.sqrt(rec_xi**2 + rec_yi**2)
        style_idx = i % len(colors)
        # Absolute value plot
        axes[0, 0].plot(rec_distance, abs_data,
                     linestyle=linestyles[style_idx],
                     marker=markers[style_idx],
                     markersize=marker_size,
                     color=colors[style_idx],
                     label=label,
                     linewidth=line_width)
        # Phase plot
        axes[0, 1].plot(rec_distance, phase_data,
                     linestyle=linestyles[style_idx],
                     marker=markers[style_idx],
                     markersize=marker_size,
                     color=colors[style_idx],
                     label=label,
                     linewidth=line_width)
        
        # Real part plot
        axes[1, 0].plot(rec_distance, real_data_list[i],
                     linestyle=linestyles[style_idx],
                     marker=markers[style_idx],
                     markersize=marker_size,
                     color=colors[style_idx],
                     label=label,
                     linewidth=line_width)
        
        # Imaginary part plot
        axes[1, 1].plot(rec_distance, imag_data_list[i],
                     linestyle=linestyles[style_idx],
                     marker=markers[style_idx],
                     markersize=marker_size,
                     color=colors[style_idx],
                     label=label,
                     linewidth=line_width)

    # Analytical solution
    rec_distance_an = np.array(y_an) if type_rec == 'Broadside' else np.array(x_an) if type_rec == 'Endfire' else np.array(x_an)
    axes[0, 0].plot(rec_distance_an, abs_an, linestyle='-', color='black', label='Analytical Solution', linewidth=2)
    axes[0, 1].plot(rec_distance_an, phase_an, linestyle='-', color='black', label='Analytical Solution', linewidth=2)
    axes[1, 0].plot(rec_distance_an, real_an, linestyle='-', color='black', label='Analytical Real Part', linewidth=2)
    axes[1, 1].plot(rec_distance_an[2:], imag_an[2:], linestyle='-', color='black', label='Analytical Imaginary Part', linewidth=2)

    axes[0, 0].set_xlabel(f'{type_string} (m)', fontsize=12)
    axes[0, 0].set_ylabel(f'Abs({component})', fontsize=12)
    axes[0, 0].set_title(f'Absolute Value of {component} Field Component', fontsize=18)
    axes[0, 0].set_yscale('log')
    axes[0, 0].grid(True)
    axes[0, 0].legend(loc='upper right')

    axes[0, 1].set_xlabel(f'{type_string} (m)', fontsize=12)
    axes[0, 1].set_ylabel(f'Phase({component}) (degrees)', fontsize=12)
    axes[0, 1].set_title(f'Phase of {component} Field Component', fontsize=18)
    axes[0, 1].grid(True)

    axes[1, 0].set_xlabel(f'{type_string} (m)', fontsize=12)
    axes[1, 0].set_ylabel(f'Re({component})', fontsize=12)
    axes[1, 0].set_title(f'Real Part of {component} Field Component', fontsize=18)
    axes[1, 0].grid(True)

    axes[1, 1].set_xlabel(f'{type_string} (m)', fontsize=12)
    axes[1, 1].set_ylabel(f'Im({component})', fontsize=12)
    axes[1, 1].set_title(f'Imaginary Part of {component} Field Component', fontsize=18)
    axes[1, 1].grid(True)

    plt.subplots_adjust(top=0.82)
    title = r'Two-Layered Model Response - $\epsilon_r = 4, 9$' \
            '\n{} Receiver Line Data of {} Component'.format(type_rec, component)
    plt.suptitle(title, fontsize=20, fontweight='bold')
    plt.tight_layout()
    plot_name = f'{index}_{purpose}_{type_rec}.png'
    out_path = os.path.join(postprocess_folder, plot_name)
    plt.savefig(out_path, dpi=300)
    plt.close()

# Prepare data for each receiver type
rec_x_bs_list, rec_y_bs_list, abs_Ex_bs_list, phase_Ex_bs_list, re_Ex_bs_list, im_Ex_bs_list = [], [], [], [], [], []
rec_x_ef_list, rec_y_ef_list, abs_Ex_ef_list, phase_Ex_ef_list, re_Ex_ef_list, im_Ex_ef_list = [], [], [], [], [], []
rec_x_ob_list, rec_y_ob_list, abs_Ex_ob_list, phase_Ex_ob_list, re_Ex_ob_list, im_Ex_ob_list = [], [], [], [], [], []

for e_data_elfe in e_data_elfe_all:

    rec_x_elfe = e_data_elfe[:, 0]
    rec_y_elfe = e_data_elfe[:, 1]

    real_Ex_elfe = e_data_elfe[:, 4]
    imag_Ex_elfe = e_data_elfe[:, 5]
    abs_Ex_elfe = np.abs(real_Ex_elfe + 1j * imag_Ex_elfe)
    phase_Ex_elfe = np.angle(real_Ex_elfe + 1j * imag_Ex_elfe) # returns phase in radians (−π to π)

    # Broadside
    i_rx_bs = num_rec_ef
    rec_x_bs = rec_x_elfe[i_rx_bs:i_rx_bs + num_rec_bs]
    rec_y_bs = rec_y_elfe[i_rx_bs:i_rx_bs + num_rec_bs]
    abs_Ex_bs = abs_Ex_elfe[i_rx_bs:i_rx_bs + num_rec_bs]
    phase_Ex_bs = phase_Ex_elfe[i_rx_bs:i_rx_bs + num_rec_bs]
    re_Ex_bs = real_Ex_elfe[i_rx_bs:i_rx_bs + num_rec_bs]
    im_Ex_bs = imag_Ex_elfe[i_rx_bs:i_rx_bs + num_rec_bs]
    rec_x_bs_list.append(rec_x_bs)
    rec_y_bs_list.append(rec_y_bs)
    abs_Ex_bs_list.append(abs_Ex_bs)
    phase_Ex_bs_list.append(phase_Ex_bs)
    re_Ex_bs_list.append(re_Ex_bs)
    im_Ex_bs_list.append(im_Ex_bs)


    # Endfire
    i_rx_ef = 0
    rec_x_ef = rec_x_elfe[i_rx_ef:i_rx_ef + num_rec_ef]
    rec_y_ef = rec_y_elfe[i_rx_ef:i_rx_ef + num_rec_ef]
    abs_Ex_ef = abs_Ex_elfe[i_rx_ef:i_rx_ef + num_rec_ef]
    phase_Ex_ef = phase_Ex_elfe[i_rx_ef:i_rx_ef + num_rec_ef]
    re_Ex_ef = real_Ex_elfe[i_rx_ef:i_rx_ef + num_rec_ef]
    im_Ex_ef = imag_Ex_elfe[i_rx_ef:i_rx_ef + num_rec_ef]
    rec_x_ef_list.append(rec_x_ef)
    rec_y_ef_list.append(rec_y_ef)
    abs_Ex_ef_list.append(abs_Ex_ef)
    phase_Ex_ef_list.append(phase_Ex_ef)
    re_Ex_ef_list.append(re_Ex_ef)
    im_Ex_ef_list.append(im_Ex_ef)

    # Oblique
    i_rx_ob = num_rec_bs + num_rec_ef
    rec_x_ob = rec_x_elfe[i_rx_ob:i_rx_ob + num_rec_ob]
    rec_y_ob = rec_y_elfe[i_rx_ob:i_rx_ob + num_rec_ob]
    abs_Ex_ob = abs_Ex_elfe[i_rx_ob:i_rx_ob + num_rec_ob]
    phase_Ex_ob = phase_Ex_elfe[i_rx_ob:i_rx_ob + num_rec_ob]
    re_Ex_ob = real_Ex_elfe[i_rx_ob:i_rx_ob + num_rec_ob]
    im_Ex_ob = imag_Ex_elfe[i_rx_ob:i_rx_ob + num_rec_ob]
    rec_x_ob_list.append(rec_x_ob)
    rec_y_ob_list.append(rec_y_ob)
    abs_Ex_ob_list.append(abs_Ex_ob)
    phase_Ex_ob_list.append(phase_Ex_ob)
    re_Ex_ob_list.append(re_Ex_ob)
    im_Ex_ob_list.append(im_Ex_ob)

# Analytical data
# -------------------------------------
# Got the broadside and endire mixed up, so now we have to fix it by re-assigning to the right
# variables.
# -------------------------------------
x_an_ef = r[1:]
y_an_ef = np.zeros_like(r[1:])
abs_Ex_an_ef = abs_endfire[1:]
phase_Ex_an_ef = phase_endfire[1:]
real_Ex_an_ef = real_endfire[1:]
imag_Ex_an_ef = imag_endfire[1:]

x_an_bs = np.zeros_like(r[1:])
y_an_bs = r[1:]
abs_Ex_an_bs = abs_broadside[1:]
phase_Ex_an_bs = phase_broadside[1:]
real_Ex_an_bs = real_broadside[1:]
imag_Ex_an_bs = imag_broadside[1:]

x_an_ob = r[1:]
y_an_ob = r[1:]
abs_Ex_an_ob = abs_45deg[1:]
phase_Ex_an_ob = phase_45deg[1:]
real_Ex_an_ob = real_45deg[1:]
imag_Ex_an_ob = imag_45deg[1:]

# Plot all on the same axes for each receiver type
plot_component_receiver_line('Ex', rec_x_ef_list, rec_y_ef_list, abs_Ex_ef_list, phase_Ex_ef_list, re_Ex_ef_list, im_Ex_ef_list,
                             x_an_ef, y_an_ef, abs_Ex_an_ef, phase_Ex_an_ef, real_Ex_an_ef, imag_Ex_an_ef, 'Endfire', 4, '4_9_emp_mesh', labels)
